# Predicting Sustained Snow Disappearance from a 1 March Forecast Origin

CODS-622 Machine Learning, course project.

**Run order:** top to bottom. Every cell reads from files committed to the repository, so
nothing here depends on the NRCS service being reachable.

**Sections map to the rubrics:** setup, data and audit, target, features and leakage,
splits, baselines, models, evaluation, ablations, demo.

## 0. Setup

Clones the repository and pins dependencies. Skip the clone if running locally.

In [ ]:
# Colab only. Detected automatically, so this cell is safe to run locally and
# safe to re-run in a session that has already cloned the repository.
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("src").is_dir():
    if not Path("snow-disappearance").is_dir():
        !git clone -q https://github.com/AlShamsiK/snow-disappearance.git
    %cd snow-disappearance
    !pip install -r requirements.txt
elif not IN_COLAB:
    print("Not running in Colab; using the local checkout.")

In [ ]:
import sys, platform, json
import numpy as np
import pandas as pd
import sklearn, lightgbm

# Locate the repository root whether the kernel started there or in notebooks/.
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src import config, clean, target, features, models, evaluate

np.random.seed(config.RANDOM_SEED)

print("python:", platform.python_version())
print("numpy:", np.__version__, "| pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__, "| lightgbm:", lightgbm.__version__)
print()
print("configuration:")
for k, v in config.as_dict().items():
    print(f"  {k} = {v}")

## 1. Data

**Source.** USDA NRCS National Water and Climate Center, SNOTEL network, pulled from the
AWDB REST service on 2026-09-19 by `src/data_download.py`. The files in `data/raw/` are
exactly what the service returned and are committed, so this notebook never touches the
network.

**Stations.** All active SNOTEL stations in the 11 western states whose record starts on or
before 1 October 1990 were candidates (548). Up to six per state were drawn with a fixed seed,
giving 66 stations. Stratifying by state spreads the sample across snow climates, from Arizona
to Washington, instead of letting Montana and Colorado dominate.

**Period.** Water years 1991 to 2025, where a water year runs 1 October to 30 September.

**Variables.** One row per station and day. Each value comes with the NRCS quality-control flag
(`V` = validated).

| Code | Meaning | Raw unit | Cleaned unit |
|------|---------|----------|--------------|
| `WTEQ` | Snow water equivalent (SWE), water in the snowpack | inches | mm |
| `SNWD` | Snow depth | inches | mm |
| `PREC` | Precipitation accumulated since 1 October | inches | mm (and daily increment `PRCP`) |
| `TAVG`, `TMAX`, `TMIN` | Daily air temperature | °F | °C |

`WTEQ` is the variable the whole project rests on: the target is built from it and the most
important feature is its value on 1 March.

In [ ]:
raw, stations, elements = clean.load_raw()
print(f"raw daily rows : {len(raw):,}")
print(f"stations       : {raw['station'].nunique()}")
print(f"date range     : {raw['date'].min().date()} to {raw['date'].max().date()}")
print(f"columns        : {list(raw.columns)}")
raw.head()

### 1.1 Summary of the data

What is actually in the file: which stations, where they are, how much of each variable is
present, and what the values look like. Coverage differs by element because SNOTEL sites
added sensors over time; snow depth in particular only became standard in the late 1990s.

In [ ]:
# Station table: where the 66 sites are and how high they sit.
station_table = (stations[["station", "name", "stateCode", "elevation_m", "latitude", "longitude", "beginDate"]]
                 .rename(columns={"stateCode": "state", "beginDate": "record_start"}))
station_table["record_start"] = station_table["record_start"].str[:10]
print("stations per state:")
print(station_table["state"].value_counts().sort_index().to_string())
print(f"\nelevation: {station_table['elevation_m'].min():.0f} to {station_table['elevation_m'].max():.0f} m "
      f"(median {station_table['elevation_m'].median():.0f} m)")
station_table.head(10)

In [ ]:
# Element coverage: share of station-days with a value, and the period of record per element.
value_cols = [c for c in config.ELEMENTS if c in raw]
coverage = pd.DataFrame({
    "present_share": raw[value_cols].notna().mean().round(3),
    "stations_reporting": {c: raw.loc[raw[c].notna(), "station"].nunique() for c in value_cols},
    "earliest_record": elements.groupby("element")["begin_date"].min().str[:10].reindex(value_cols),
    "latest_start": elements.groupby("element")["begin_date"].max().str[:10].reindex(value_cols),
    "raw_unit": elements.groupby("element")["stored_unit"].first().reindex(value_cols),
})
display(coverage)
print("\nraw value distribution (raw units):")
raw[value_cols].describe().round(2).T

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [2.2, 1]})

# Left: SWE at one station across five water years. The annual build-up and melt-out is the
# signal we are predicting; the melt-out date is where each curve returns to zero.
example_station = config.STATIONS[12]
sub = raw[(raw["station"] == example_station) & (raw["date"] >= "2015-10-01") & (raw["date"] <= "2020-09-30")]
axes[0].plot(sub["date"], sub["WTEQ"] * config.IN_TO_MM, lw=1)
axes[0].set_title(f"Snow water equivalent, {stations.set_index('station').loc[example_station, 'name']} ({example_station})")
axes[0].set_ylabel("SWE (mm)")
for wy in range(2016, 2021):
    axes[0].axvline(pd.Timestamp(f"{wy}-03-01"), color="grey", ls=":", lw=0.8)
axes[0].text(0.01, 0.95, "dotted: 1 March forecast origin", transform=axes[0].transAxes, va="top", fontsize=9)

# Right: elevation of the station sample.
axes[1].hist(station_table["elevation_m"], bins=15, edgecolor="white")
axes[1].set_title("Station elevation")
axes[1].set_xlabel("m above sea level")
axes[1].set_ylabel("stations")
plt.tight_layout()
plt.show()

### 1.2 Data audit

Before any cleaning: how much is missing, are there duplicate station-date rows, are the
identifiers consistent, and are there values that cannot be physically right. Nothing is
modified here; the table is the evidence for the cleaning decisions that follow.

The last rows of the table count values whose NRCS quality-control flag is not `V` (validated).
The common alternative flags are `E`, a value NRCS itself estimated or edited to fill a gap, and
`C`/`K`, values NRCS adjusted after comparison with neighbouring sensors. These are the provider's
own gap-filling and are kept, but they are counted here because a target date that falls inside
a long run of estimated values is less certain than one that falls on validated readings.


In [ ]:
audit = clean.audit_raw(raw)
audit

In [ ]:
# Where is SWE missing? Share of missing days per station and water year.
miss = raw.assign(wy=clean.water_year(raw["date"]))
miss = miss.groupby(["station", "wy"])["WTEQ"].apply(lambda x: x.isna().mean()).unstack("wy")

fig, ax = plt.subplots(figsize=(13, 9))
im = ax.imshow(miss.to_numpy(), aspect="auto", cmap="Reds", vmin=0, vmax=1)
ax.set_yticks(range(len(miss.index)), miss.index, fontsize=6)
ax.set_xticks(range(len(miss.columns)), miss.columns, rotation=90, fontsize=7)
ax.set_title("Share of days with SWE missing, by station (rows) and water year (columns)")
plt.colorbar(im, ax=ax, fraction=0.02, label="missing share")
plt.tight_layout()
plt.show()
print(f"station-years with any SWE missing: {(miss > 0).sum().sum()} of {miss.size}")
print(f"station-years with > 10% SWE missing: {(miss > 0.10).sum().sum()}")

### 1.3 Cleaning and screening

Each rule below is applied by `src/clean.py`, and each one records how many rows or values it
touched, so the write-up can justify decisions with numbers rather than assert them. The steps,
in order:

1. **Complete calendar.** Every station gets every day from 1 Oct 1990 to 30 Sep 2025, so a
   day the service did not return becomes an explicit missing value instead of a silent hole.
2. **Duplicates.** Repeated station-date rows are dropped, keeping the first.
3. **Units.** Inches to millimetres, Fahrenheit to Celsius.
4. **Physical limits.** SWE, depth and precipitation slightly below zero are snow pillow drift
   and are clipped to zero; strongly negative values are sensor faults and become missing.
   Temperatures outside −50 to 50 °C become missing, as do days where the minimum exceeds the
   maximum.
5. **Daily precipitation.** `PREC` accumulates over the water year, so the daily increment
   `PRCP` is derived; negative increments are gauge drift or resets, not negative rain, and
   become zero.
6. **SWE anomalies.** Known snow pillow failure modes: an isolated one-day spike and an
   isolated zero inside a continuous snowpack are both removed. Flat-lined stretches are
   flagged but kept, because a frozen pack can legitimately hold a constant value for weeks.
7. **Short gaps.** Linear interpolation across gaps of at most five days for SWE, depth and
   temperature. Longer gaps stay missing. Precipitation is never interpolated.
8. **Screening.** A station-year is dropped when SWE coverage is below 90% in either the
   feature window (Oct to Feb) or the target window (Mar to Sep), or when peak SWE is below
   50 mm, meaning no real snowpack formed. A station is dropped when fewer than 20 usable
   years remain.

The thresholds live in `src/config.py` so they can be varied in the ablation section.

In [ ]:
daily, cleaning_log, exclusions = clean.run_pipeline(raw)
print(f"raw rows: {len(raw):,}  ->  clean rows: {len(daily):,}  "
      f"({daily['station'].nunique()} stations, {daily.groupby(['station', 'water_year']).ngroups} station-years)")
pd.set_option("display.max_colwidth", 80)
cleaning_log

In [ ]:
# Why station-years were excluded, and how many usable years each station keeps.
print("exclusions by reason:")
print(exclusions["reason"].value_counts().to_string())
kept = daily.groupby("station")["water_year"].nunique().rename("usable_years")
print(f"\nusable years per station: min {kept.min()}, median {kept.median():.0f}, max {kept.max()}")
exclusions.head(10)

In [ ]:
# Before and after for the station-year where cleaning changed the most SWE values.
# A removed spike or dropout is a one-day gap, which the short-gap rule then refills by
# interpolation, so "changed" means removed or replaced, not only left missing.
raw_si = clean.to_si(clean.complete_calendar(raw))
raw_si["water_year"] = clean.water_year(raw_si["date"])
merged = raw_si.merge(daily[["station", "date", "WTEQ"]], on=["station", "date"], how="inner", suffixes=("_raw", "_clean"))
changed = merged["WTEQ_raw"].notna() & (merged["WTEQ_clean"].isna() | ((merged["WTEQ_clean"] - merged["WTEQ_raw"]).abs() > 1e-6))
print(f"SWE values changed by cleaning: {changed.sum()} of {merged['WTEQ_raw'].notna().sum():,}")
if changed.any():
    st, wy = merged.loc[changed].groupby(["station", "water_year"]).size().idxmax()
    sel = (merged["station"] == st) & (merged["water_year"] == wy)
    ex = merged[sel]
    # zoom to the weeks around the changed values so the difference is visible
    lo, hi = merged.loc[changed & sel, "date"].agg(["min", "max"])
    ex = ex[(ex["date"] >= lo - pd.Timedelta(days=21)) & (ex["date"] <= hi + pd.Timedelta(days=21))]
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(ex["date"], ex["WTEQ_raw"], color="tab:red", lw=1.6, label="raw")
    ax.plot(ex["date"], ex["WTEQ_clean"], color="tab:blue", lw=1, label="clean")
    ax.scatter(ex.loc[changed[sel], "date"], ex.loc[changed[sel], "WTEQ_raw"], color="tab:red", s=18, zorder=3, label="changed")
    ax.set_title(f"Effect of cleaning on SWE, {st}, water year {wy}, zoomed ({int(changed[sel].sum())} values changed)")
    ax.set_ylabel("SWE (mm)"); ax.legend()
    plt.tight_layout(); plt.show()

## 2. Target: sustained snow disappearance

**The raw data carry no label.** NRCS publishes measurements, not melt-out dates. The label is
constructed from the SWE series, which makes this a supervised regression problem with a
derived target, and the construction rule is a modelling decision that has to be stated and
tested rather than taken as given.

**Definition.** For each station and water year, the target is the first day on or after
1 March when SWE is at or below 2.5 mm and stays there for the next 14 consecutive days.
It is expressed as days after 1 March, so it is comparable across leap and non-leap years.

The "sustained" requirement is what distinguishes a genuine melt-out from a brief bare-ground
spell, a sensor dropout, or a late-season snowfall that vanishes within days. Both parameters
are configurable and their influence is tested in the ablation section.

Station-years where the pack never disappears, where it was already gone on 1 March, or where
a data gap sits before the melt-out date are recorded with a reason rather than dropped
silently. Only rows with reason `melt_out` are used for training.

In [ ]:
targets = target.build_targets(daily)
print("station-years by outcome:")
print(targets["reason"].value_counts().to_string())
labelled = targets[targets["reason"] == "melt_out"]
print(f"\nusable labelled station-years: {len(labelled)} of {len(targets)}")
print(f"target (days after 1 March): mean {labelled['days_from_march1'].mean():.1f}, "
      f"sd {labelled['days_from_march1'].std():.1f}, "
      f"range {labelled['days_from_march1'].min():.0f} to {labelled['days_from_march1'].max():.0f}")
targets.head(8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 2]})

# Left: distribution of the target.
axes[0].hist(labelled["days_from_march1"], bins=30, edgecolor="white")
axes[0].set_xlabel("days after 1 March"); axes[0].set_ylabel("station-years")
axes[0].set_title("Sustained snow disappearance")

# Right: one station-year with the origin and the label marked.
row = labelled.sample(1, random_state=config.RANDOM_SEED).iloc[0]
ex = daily[(daily["station"] == row["station"]) & (daily["water_year"] == row["water_year"])]
axes[1].plot(ex["date"], ex["WTEQ"], lw=1.2)
axes[1].axvline(target.forecast_origin(row["water_year"]), color="grey", ls=":", label="1 March origin")
axes[1].axvline(row["disappearance_date"], color="tab:red", ls="--", label=f"label: {row['disappearance_date'].date()} (+{row['days_from_march1']:.0f} d)")
axes[1].axhline(config.SWE_ZERO_THRESHOLD, color="tab:red", lw=0.6, alpha=0.5)
axes[1].set_title(f"{row['station']}, water year {row['water_year']}"); axes[1].set_ylabel("SWE (mm)")
axes[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
# Persist the processed data so later sections and the report use exactly these tables.
daily.to_csv(config.DATA_PROCESSED / "daily_clean.csv.gz", index=False, compression="gzip")
targets.to_csv(config.DATA_PROCESSED / "targets.csv", index=False)
cleaning_log.to_csv(config.DATA_PROCESSED / "cleaning_log.csv", index=False)
exclusions.to_csv(config.DATA_PROCESSED / "exclusions.csv", index=False)
print("written to", config.DATA_PROCESSED)

## 3. Features as of 1 March

**Leakage rule:** a feature may use data dated strictly before 1 March of its own water year,
static station attributes, or statistics estimated on training water years only. Nothing else.

Three traps specific to this problem:
1. Using any observation from March onward, which is the whole forecasting task.
2. Station climatology computed over the full record, including the test years.
3. Percent-of-median fields taken from the data source, whose medians are computed over a
   period of record that includes the test years.

In [ ]:
# X = features.build_feature_table()
# features.assert_no_leakage(X, daily)
# X.head()

## 4. Splits

Random k-fold would be wrong here. Station-years are correlated through time and across
nearby stations, so a random split leaks both the future and the neighbours.

Three evaluation regimes:
- **Forward chaining over water years.** Train on the past, predict forward. This is the
  operational question.
- **Leave-one-station-out.** Can the model handle a station it has never seen.
- **Spatial blocks.** Hold out whole regions, to check that apparent skill is not just
  spatial autocorrelation.

The most recent block of water years is held out entirely and touched once, at the end.

In [ ]:
# splits = evaluate.forward_chaining_splits(X)

## 5. Baselines

The model has to beat these or the project has no finding.

- **Climatology.** Each station's historical average disappearance date, from training years only.
- **Degree-day melt model.** Physically motivated: melt the 1 March snowpack at a calibrated rate.
- **SWE-only linear regression.** How much of the answer is simply how much snow is on the ground.

In [ ]:
# clim = models.ClimatologyBaseline().fit(X_train, y_train)
# dd = models.DegreeDayBaseline().fit(X_train, y_train)
# swe_lin = models.SWEOnlyLinearBaseline().fit(X_train, y_train)

## 6. Models

Ridge, random forest, and gradient boosting, with hyperparameters tuned inside the training
folds only. Quantile regressors supply the prediction interval used in the demo.

## 7. Evaluation

Headline metrics in days: MAE, RMSE, and signed bias. The number that matters most is the
skill score against climatology, because an MAE that sounds good in isolation may be no
better than the long-run average for that station.

Bias is reported separately from MAE: a model that is unbiased on average but systematically
late in warm springs is failing differently from one that is simply noisy.

## 8. Ablations and robustness

- Feature groups: snowpack only, plus accumulation-season weather, plus static attributes.
- Sensitivity to the target definition: vary the sustain window and the SWE threshold.
- Error by elevation band.
- Performance in low-snow versus high-snow years.
- Generalisation to unseen stations and unseen years.

### Failure cases to look for
Low-elevation stations with intermittent snowpack, late-season storms, rain-on-snow events,
unusually warm springs, and stations near the rain-snow transition.

## 9. Demo

Pick a station and a water year, show the features exactly as they stood on 1 March, predict
the disappearance date with an interval, and compare it against what actually happened.

The model is loaded from `models/`, not retrained.

In [ ]:
import joblib

# model = joblib.load(config.MODELS / "model.joblib")

STATION = None      # e.g. "301:CA:SNTL"
WATER_YEAR = None   # e.g. 2019

# 1. Show the inputs available on 1 March of that water year
# 2. Predict the disappearance date and interval
# 3. Show the observed date and the error in days

## 10. Limitations

To be written up alongside the report. Candidates: the target definition is a choice and
results move with it, the station set is a subset rather than the full network, prediction
skill will degrade at low elevations, and the model is fitted to historical climate and
carries no guarantee under conditions outside the training record.